# CMS Open Payments General Payments Analysis

**Course:** CHIP 705  
**Project:** Python-based data analysis project  
**Research question:** How do general industry payments to physicians vary by medical specialty, nature of payment, and state?

This notebook analyzes a cleaned sample extracted from the CMS Open Payments General Payments dataset. The original full dataset was very large, so this project uses a cleaned physician-payment dataset with only the columns needed for the research question.


## 1. Import packages

This project uses `pandas` for dataframe work and `matplotlib` for visualizations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("data/cleaned_general_payments_sample.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


## 2. Load the cleaned dataset

The cleaned dataset includes the columns needed to answer the research question: recipient type, specialty, state, nature of payment, payment amount, and payment date.


In [ ]:
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()


## 3. Clean and prepare the data

The project focuses on physicians only. Payment amount is converted to numeric and payment date is converted to a date variable.


In [ ]:
# Standardize column names by keeping the original CMS-style names
needed_columns = [
    "Covered_Recipient_Type",
    "Covered_Recipient_Specialty_1",
    "Recipient_State",
    "Nature_of_Payment_or_Transfer_of_Value",
    "Total_Amount_of_Payment_USDollars",
    "Date_of_Payment"
]

df = df[needed_columns].copy()

# Keep physician payments only
df = df[df["Covered_Recipient_Type"] == "Covered Recipient Physician"].copy()

# Convert data types
df["Total_Amount_of_Payment_USDollars"] = pd.to_numeric(
    df["Total_Amount_of_Payment_USDollars"], errors="coerce"
)
df["Date_of_Payment"] = pd.to_datetime(df["Date_of_Payment"], errors="coerce")

# Remove rows missing key analysis fields
df = df.dropna(
    subset=[
        "Covered_Recipient_Specialty_1",
        "Recipient_State",
        "Nature_of_Payment_or_Transfer_of_Value",
        "Total_Amount_of_Payment_USDollars",
        "Date_of_Payment"
    ]
)

print(df.shape)
df.info()


## 4. Descriptive overview

In [ ]:
overview = pd.DataFrame({
    "metric": [
        "Number of payment records",
        "Number of specialties",
        "Number of states",
        "Number of payment categories",
        "Total payment amount",
        "Median payment amount"
    ],
    "value": [
        len(df),
        df["Covered_Recipient_Specialty_1"].nunique(),
        df["Recipient_State"].nunique(),
        df["Nature_of_Payment_or_Transfer_of_Value"].nunique(),
        round(df["Total_Amount_of_Payment_USDollars"].sum(), 2),
        round(df["Total_Amount_of_Payment_USDollars"].median(), 2)
    ]
})

overview


## 5. Analysis 1: Which specialties receive the highest total and median payments?

This section summarizes total payment amount, median payment amount, and number of payments by specialty.


In [ ]:
specialty_summary = (
    df.groupby("Covered_Recipient_Specialty_1")
    .agg(
        total_payment=("Total_Amount_of_Payment_USDollars", "sum"),
        median_payment=("Total_Amount_of_Payment_USDollars", "median"),
        number_of_payments=("Total_Amount_of_Payment_USDollars", "count")
    )
    .reset_index()
    .sort_values("total_payment", ascending=False)
)

specialty_summary.head(10)


In [ ]:
top_specialties = specialty_summary.head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_specialties["Covered_Recipient_Specialty_1"], top_specialties["total_payment"])
plt.gca().invert_yaxis()
plt.title("Top 10 Specialties by Total General Payment Amount")
plt.xlabel("Total Payment Amount (USD)")
plt.ylabel("Specialty")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "top_specialties_chart.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Analysis 2: Which payment categories account for the largest number of payments and largest total amounts?

This section summarizes payments by nature-of-payment category.


In [ ]:
nature_summary = (
    df.groupby("Nature_of_Payment_or_Transfer_of_Value")
    .agg(
        total_payment=("Total_Amount_of_Payment_USDollars", "sum"),
        median_payment=("Total_Amount_of_Payment_USDollars", "median"),
        number_of_payments=("Total_Amount_of_Payment_USDollars", "count")
    )
    .reset_index()
    .sort_values("total_payment", ascending=False)
)

nature_summary.head(10)


In [ ]:
top_nature = nature_summary.head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_nature["Nature_of_Payment_or_Transfer_of_Value"], top_nature["total_payment"])
plt.gca().invert_yaxis()
plt.title("Top Nature-of-Payment Categories by Total Amount")
plt.xlabel("Total Payment Amount (USD)")
plt.ylabel("Nature of Payment")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "payment_category_chart.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Analysis 3: How do North Carolina payments compare with other states?

This section compares North Carolina physician payment records with physician payment records from all other states in the cleaned dataset.


In [ ]:
df["NC_vs_Other"] = np.where(df["Recipient_State"] == "NC", "North Carolina", "Other states")

nc_summary = (
    df.groupby("NC_vs_Other")
    .agg(
        total_payment=("Total_Amount_of_Payment_USDollars", "sum"),
        median_payment=("Total_Amount_of_Payment_USDollars", "median"),
        number_of_payments=("Total_Amount_of_Payment_USDollars", "count")
    )
    .reset_index()
)

nc_summary


In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(nc_summary["NC_vs_Other"], nc_summary["median_payment"])
plt.title("Median General Payment: North Carolina vs Other States")
plt.xlabel("Recipient location")
plt.ylabel("Median Payment Amount (USD)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nc_comparison_chart.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Save output tables

In [ ]:
specialty_summary.to_csv(OUTPUT_DIR / "specialty_summary.csv", index=False)
nature_summary.to_csv(OUTPUT_DIR / "nature_of_payment_summary.csv", index=False)
nc_summary.to_csv(OUTPUT_DIR / "nc_vs_other_summary.csv", index=False)

print("Output files saved in:", OUTPUT_DIR)


## 9. Interpretation

The analysis shows that general payment patterns are not evenly distributed across specialties or payment categories. A small number of specialties account for a large share of total payment dollars. Payment categories also differ: some categories may have many small payments, while others may have fewer but larger payments.

The North Carolina comparison provides a geographic view of the data. The median payment amount is used because payment data can be skewed by very large individual payments.

## Limitations

This notebook uses a cleaned sample extracted from a much larger CMS Open Payments file. Because it is a sample, the results should be interpreted as a demonstration of the Python analysis workflow rather than as the final national totals from the full CMS dataset. The analysis also focuses on general payment records and does not measure whether payments affected clinical decisions or patient outcomes.
